In [ ]:
import asyncio
import aiohttp
import pandas as pd
import datetime
import math

1. Obtenemos los datos de la api

In [2]:
async def fetch(session, params):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "es-MX,es;q=0.9",
        "Referer": "https://www.datos.gob.mx/",
    }
    url = "https://www.datos.gob.mx/api/3/action/current_package_list_with_resources"
    async with session.get(url, headers=headers, params=params) as response:
        if response.status == 200:
            result = await response.json()
            return result['result']
        else:
            return response.error

async def multiFetch():
    maxExpectedPackages = 2500
    block_size = 100
    numRequests = math.ceil(maxExpectedPackages / block_size) 
    params = list()
    conjuntos = []
    for i in range(numRequests):
        if i == 0:
            param = {"limit" : block_size - 1 , "offset" : i * block_size}
        else:
            param = {"limit" : block_size -1 , "offset" : i * block_size}
        params.append(param)
    #print(params)
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, param) for param in params]
        resultados = await asyncio.gather(*tasks, return_exceptions=True)

    for resultado in resultados:
        conjuntos += resultado

    return conjuntos

In [3]:
def iterar_recursos(conjuntos):
    recursos = []
    for conjunto in conjuntos:
        for recurso in conjunto["resources"]:
            datum = {
                "nombre_categoria": conjunto["groups"][0]["display_name"],
                "clave_categoria": conjunto["groups"][0]["name"],
                "descripcion_categoria": conjunto["groups"][0]["description"],
                "id_categoria": conjunto["groups"][0]["id"],
                "nombre_institucion": conjunto["organization"]["title"],
                "descripcion_institucion": conjunto["organization"]["description"],
                "nombre_paquete": conjunto["title"],
                "clave_paquete": conjunto["name"],
                "notas_paquete": conjunto["notes"],
                "etiquetas": list(map(lambda x: x["name"], conjunto["tags"])),
                "id_paquete": conjunto["id"],
                "id_recurso": recurso["id"],
                "nombre_recurso": recurso["name"],
                "descripcion_recurso": recurso["description"],
                "url_recurso": recurso["url"],
                "frecuencia_actualizacion": recurso["update_frequency"],
                "creacion_recurso": recurso["created"],
                "ultima_modificacion": recurso["last_modified"],
                "modificacion_metadatos": recurso["metadata_modified"]
            }
            recursos.append(datum)
    return recursos

In [4]:
data_ckan = await multiFetch()
len(data_ckan)

1735

In [5]:
conjuntos_df = pd.DataFrame(data_ckan)
conjuntos_df.columns

Index(['author', 'author_email', 'creator_user_id', 'id', 'isopen',
       'license_id', 'license_title', 'license_url', 'maintainer',
       'maintainer_email', 'metadata_created', 'metadata_modified', 'name',
       'notes', 'num_resources', 'num_tags', 'organization', 'owner_org',
       'private', 'state', 'title', 'type', 'url', 'version', 'groups',
       'resources', 'tags', 'extras', 'relationships_as_subject',
       'relationships_as_object'],
      dtype='str')

2. Iteramos sobre los recursos para obtener su información

In [6]:
recursos_ckan = iterar_recursos(data_ckan)
len(recursos_ckan)

7772

In [7]:
recursos_df = pd.DataFrame(recursos_ckan)
recursos_df.shape

(7772, 19)

In [8]:
recursos_df.columns

Index(['nombre_categoria', 'clave_categoria', 'descripcion_categoria',
       'id_categoria', 'nombre_institucion', 'descripcion_institucion',
       'nombre_paquete', 'clave_paquete', 'notas_paquete', 'etiquetas',
       'id_paquete', 'id_recurso', 'nombre_recurso', 'descripcion_recurso',
       'url_recurso', 'frecuencia_actualizacion', 'creacion_recurso',
       'ultima_modificacion', 'modificacion_metadatos'],
      dtype='str')

In [9]:
recursos_df.dtypes

nombre_categoria               str
clave_categoria                str
descripcion_categoria          str
id_categoria                   str
nombre_institucion             str
descripcion_institucion        str
nombre_paquete                 str
clave_paquete                  str
notas_paquete                  str
etiquetas                   object
id_paquete                     str
id_recurso                     str
nombre_recurso                 str
descripcion_recurso            str
url_recurso                    str
frecuencia_actualizacion       str
creacion_recurso               str
ultima_modificacion            str
modificacion_metadatos         str
dtype: object

In [10]:
recursos_df.loc[0]

nombre_categoria                                                    Seguridad
clave_categoria                                                     seguridad
descripcion_categoria       Datos a nivel federal y estatal sobre los deli...
id_categoria                             98157bd6-8f5e-4da4-a14c-b2e54ba68779
nombre_institucion                      Prevención y Reinserción Social (PRS)
descripcion_institucion     Instrumenta la política penitenciaria a nivel ...
nombre_paquete              Cuaderno Mensual Estadístico Penitenciario (ju...
clave_paquete               cuaderno_mensual_estadistico_penitenciario_jun...
notas_paquete               Datos de la población privada de la libertad e...
etiquetas                   [administración justicia, centro penitenciario...
id_paquete                               28d988ea-0b26-47ed-85b9-8b3b9e54ccec
id_recurso                               575e5364-a3b8-442a-8303-abbdb4dc29b5
nombre_recurso              Estadística penitenciaria: centros, 

In [11]:
recursos_df["creacion_recurso"] = pd.to_datetime(recursos_df["creacion_recurso"])
recursos_df["ultima_modificacion"] = pd.to_datetime(recursos_df["ultima_modificacion"])
recursos_df["modificacion_metadatos"] = pd.to_datetime(recursos_df["modificacion_metadatos"])

In [12]:
recursos_df.dtypes

nombre_categoria                       str
clave_categoria                        str
descripcion_categoria                  str
id_categoria                           str
nombre_institucion                     str
descripcion_institucion                str
nombre_paquete                         str
clave_paquete                          str
notas_paquete                          str
etiquetas                           object
id_paquete                             str
id_recurso                             str
nombre_recurso                         str
descripcion_recurso                    str
url_recurso                            str
frecuencia_actualizacion               str
creacion_recurso            datetime64[us]
ultima_modificacion         datetime64[us]
modificacion_metadatos      datetime64[us]
dtype: object

In [13]:
recursos_df[recursos_df["ultima_modificacion"] == recursos_df["modificacion_metadatos"]]

,nombre_categoria,clave_categoria,descripcion_categoria,id_categoria,nombre_institucion,descripcion_institucion,nombre_paquete,clave_paquete,notas_paquete,etiquetas,id_paquete,id_recurso,nombre_recurso,descripcion_recurso,url_recurso,frecuencia_actualizacion,creacion_recurso,ultima_modificacion,modificacion_metadatos


In [14]:
recursos_df["ultima_modificacion"].isna().value_counts()

ultima_modificacion
True     7347
False     425
Name: count, dtype: int64

In [15]:
recursos_df["modificacion_metadatos"].isna().value_counts()

modificacion_metadatos
False    7772
Name: count, dtype: int64

In [16]:
recursos_df[recursos_df["modificacion_metadatos"] > recursos_df["creacion_recurso"]].shape

(7767, 19)

3. Les agregamos una columna con los años de creación

In [17]:
recursos_df["anio_creacion"] = recursos_df["creacion_recurso"].dt.year
recursos_df["anio_ultima_modificacion"] = recursos_df["ultima_modificacion"].dt.year

4. Les agregamos la semana de creación y la semana de modificación

In [18]:
recursos_df["semana_creacion"] = recursos_df["creacion_recurso"].dt.isocalendar().week
recursos_df["semana_ultima_actualizacion"] = recursos_df["ultima_modificacion"].dt.isocalendar().week

5. Generamos el csv

In [19]:
hoy = datetime.datetime.today()
hoy = pd.to_datetime(hoy)
hoy

Timestamp('2026-09-25 13:48:29.494920')

In [20]:
nombre = f"ckan_{hoy.day}_{hoy.month_name()}_{hoy.year}"
nombre

'ckan_25_September_2026'

In [142]:
recursos_df.head()

,nombre_categoria,clave_categoria,descripcion_categoria,id_categoria,nombre_institucion,descripcion_institucion,nombre_paquete,clave_paquete,notas_paquete,etiquetas,...,descripcion_recurso,url_recurso,frecuencia_actualizacion,creacion_recurso,ultima_modificacion,modificacion_metadatos,anio_creacion,anio_ultima_modificacion,semana_creacion,semana_ultima_actualizacion
0,Seguridad,seguridad,Datos a nivel federal y estatal sobre los deli...,98157bd6-8f5e-4da4-a14c-b2e54ba68779,Prevención y Reinserción Social (PRS),Instrumenta la política penitenciaria a nivel ...,Cuaderno Mensual Estadístico Penitenciario (ju...,cuaderno_mensual_estadistico_penitenciario_jun...,Datos de la población privada de la libertad e...,"[administración justicia, centro penitenciario...",...,"Número de centros penitenciarios, espacios, po...",https://repodatos.atdt.gob.mx/api_update/prs/c...,Mensual,2026-09-18 22:23:58.674833,NaT,2026-09-18 22:24:00.893006,2026,NaN,38,<NA>
1,Seguridad,seguridad,Datos a nivel federal y estatal sobre los deli...,98157bd6-8f5e-4da4-a14c-b2e54ba68779,Prevención y Reinserción Social (PRS),Instrumenta la política penitenciaria a nivel ...,Cuaderno Mensual Estadístico Penitenciario (ju...,cuaderno_mensual_estadistico_penitenciario_jun...,Datos de la población privada de la libertad e...,"[administración justicia, centro penitenciario...",...,"Espacios, sobrepoblación y población privada d...",https://repodatos.atdt.gob.mx/api_update/prs/c...,Mensual,2026-09-18 23:08:22.533736,NaT,2026-09-18 23:08:57.986306,2026,NaN,38,<NA>
2,Seguridad,seguridad,Datos a nivel federal y estatal sobre los deli...,98157bd6-8f5e-4da4-a14c-b2e54ba68779,Prevención y Reinserción Social (PRS),Instrumenta la política penitenciaria a nivel ...,Cuaderno Mensual Estadístico Penitenciario (ju...,cuaderno_mensual_estadistico_penitenciario_jun...,Datos de la población privada de la libertad e...,"[administración justicia, centro penitenciario...",...,Tipos de centros penitenciarios por entidad fe...,https://repodatos.atdt.gob.mx/api_update/prs/c...,Mensual,2026-09-18 23:33:28.629553,NaT,2026-09-18 23:33:30.688149,2026,NaN,38,<NA>
3,Seguridad,seguridad,Datos a nivel federal y estatal sobre los deli...,98157bd6-8f5e-4da4-a14c-b2e54ba68779,Prevención y Reinserción Social (PRS),Instrumenta la política penitenciaria a nivel ...,Cuaderno Mensual Estadístico Penitenciario (ju...,cuaderno_mensual_estadistico_penitenciario_jun...,Datos de la población privada de la libertad e...,"[administración justicia, centro penitenciario...",...,Ingresos y egresos de la población privada de ...,https://repodatos.atdt.gob.mx/api_update/prs/c...,Mensual,2026-09-18 23:40:22.646146,NaT,2026-09-18 23:40:24.666120,2026,NaN,38,<NA>
4,Seguridad,seguridad,Datos a nivel federal y estatal sobre los deli...,98157bd6-8f5e-4da4-a14c-b2e54ba68779,Prevención y Reinserción Social (PRS),Instrumenta la política penitenciaria a nivel ...,Cuaderno Mensual Estadístico Penitenciario (ju...,cuaderno_mensual_estadistico_penitenciario_jun...,Datos de la población privada de la libertad e...,"[administración justicia, centro penitenciario...",...,Traslados internacionales y extradiciones por ...,https://repodatos.atdt.gob.mx/api_update/prs/c...,Mensual,2026-09-18 23:46:13.136573,NaT,2026-09-18 23:46:15.338029,2026,NaN,38,<NA>


In [143]:
recursos_df.to_csv(f"./{nombre}.csv")